# Lecture: Autoencoder for Unsupervised Representation Learning

In the previous lectures, we modeled images either through explicit density estimation (Gaussian) or autoregressive factorization (PixelCNN). Both approaches aim to capture the full data distribution. However, for many practical tasks — such as compression, denoising, or visualization — we are primarily interested in learning a compact, meaningful representation of the data.

An **autoencoder** is a neural network trained to compress data into a low-dimensional **latent space** and then reconstruct it from that representation. It consists of two parts:
- **Encoder**: Maps the input $\mathbf{x} \in \mathbb{R}^N$ to a latent vector $\mathbf{z} \in \mathbb{R}^M$ with $M \ll N$.
- **Decoder**: Maps the latent vector $\mathbf{z}$ back to a reconstruction $\hat{\mathbf{x}} \in \mathbb{R}^N$.

Training minimizes the **reconstruction loss** between the input and its reconstruction — typically mean squared error (MSE). The bottleneck forces the model to learn a compressed representation that captures the most important structure in the data.

When the latent dimension is set to 2, we can directly visualize the latent space and observe how the model organizes different digit classes — without any label supervision during training.

Run the following cell only if you are working with Google Colab to copy the required .py file into the root directory. If you are working locally, ignore this cell.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/C2-Autoencoders/Autoencoder.py ./

### Data Preparation

We use the full MNIST training set (60,000 images). Each 28×28 grayscale image is normalized to the range [0, 1]. Since the autoencoder reconstructs pixel intensities as continuous values, no quantization is needed — unlike PixelCNN.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# -------------------------------------------------
# Device configuration
# -------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -------------------------------------------------
# Dataset preparation
# -------------------------------------------------

# Normalize pixel values to [0, 1]
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

### Model Architecture

We use a convolutional autoencoder. Convolutional layers exploit the spatial structure of images and are more parameter-efficient than fully connected architectures.

- **Encoder**: Three convolutional blocks (Conv → BatchNorm → ReLU) with stride 2 for downsampling, followed by a linear projection to the latent vector.
- **Decoder**: A linear projection back to the spatial representation, followed by transposed convolutions (ConvTranspose2d) for upsampling, and a final sigmoid activation to constrain outputs to [0, 1].

The latent dimension is set to 2 by default, enabling direct 2D visualization of the learned representation.

In [ ]:
from Autoencoder import Autoencoder

# Verify output shapes
_ae = Autoencoder(latent_dim=2)
_x  = torch.zeros(4, 1, 28, 28)
print("Reconstruction shape:", _ae(_x).shape)      # expected: (4, 1, 28, 28)
print("Latent shape:        ", _ae.encode(_x).shape)  # expected: (4, 2)

### Training

The autoencoder is trained to minimize the mean squared error (MSE) between input images and their reconstructions. MSE penalizes the per-pixel difference and is a natural choice when pixel intensities are continuous values in [0, 1].

Training for 10 epochs on the full MNIST training set takes approximately 2–3 minutes on a Colab GPU.

In [ ]:
import torch.optim as optim
import torch.nn.functional as F

# -------------------------------------------------
# Model initialization
# -------------------------------------------------
LATENT_DIM = 2

model = Autoencoder(latent_dim=LATENT_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# -------------------------------------------------
# Training loop
# -------------------------------------------------
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, _ in train_loader:
        x = x.to(device, non_blocking=True)

        optimizer.zero_grad()
        x_hat = model(x)
        loss = F.mse_loss(x_hat, x)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1:2d}, Loss: {total_loss / len(train_loader):.6f}")

In [ ]:
model.save_model()

If you do not want to train, you can load the pre-trained model (latent_dim=2, 10 epochs, full MNIST training set).

In [ ]:
import torch
from Autoencoder import Autoencoder

device = "cuda" if torch.cuda.is_available() else "cpu"
LATENT_DIM = 2

model = Autoencoder(latent_dim=LATENT_DIM).to(device)
model.load_model(path="AIBIP/C2-Autoencoders/models/autoencoder_mnist.pth", device=device)

### Reconstruction Quality

We evaluate the autoencoder qualitatively by comparing original test images with their reconstructions. Each pair shows what information the model successfully preserves through the 2-dimensional bottleneck.

In [ ]:
import matplotlib.pyplot as plt

model.eval()

# Grab one batch from the test set
x_batch, _ = next(iter(test_loader))
x_batch = x_batch[:10].to(device)

with torch.no_grad():
    x_hat = model(x_batch)

# Plot originals (top row) and reconstructions (bottom row)
fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i in range(10):
    axes[0, i].imshow(x_batch[i].squeeze().cpu(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(x_hat[i].squeeze().cpu(), cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original",       fontsize=10)
axes[1, 0].set_ylabel("Reconstruction", fontsize=10)

plt.tight_layout()
plt.show()

### Latent Space Visualization

Because the latent dimension is 2, we can plot every test image as a point in the 2D latent space, colored by its true digit class. This reveals how the autoencoder organizes the data — entirely without label supervision during training.

Well-separated clusters indicate that the autoencoder has learned class-discriminative features purely from the reconstruction objective.

In [ ]:
import numpy as np

model.eval()

all_z      = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:
        z = model.encode(x.to(device))
        all_z.append(z.cpu().numpy())
        all_labels.append(y.numpy())

all_z      = np.concatenate(all_z,      axis=0)  # (10000, 2)
all_labels = np.concatenate(all_labels, axis=0)  # (10000,)

# -------------------------------------------------
# Scatter plot
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))

scatter = ax.scatter(
    all_z[:, 0], all_z[:, 1],
    c=all_labels,
    cmap="tab10",
    s=2,
    alpha=0.6
)

plt.colorbar(scatter, ax=ax, label="Digit class", ticks=range(10))
ax.set_xlabel("z[0]")
ax.set_ylabel("z[1]")
ax.set_title("2D Latent Space — MNIST Test Set")
plt.tight_layout()
plt.show()

### Latent Space Interpolation

Because the latent space is continuous, we can interpolate linearly between the latent representations of two test images. The decoded images along the path reveal how the model transitions between different digit shapes, demonstrating that the latent space captures smooth, meaningful structure.

In [ ]:
model.eval()

# Pick two test images from different digit classes
test_images, test_labels = next(iter(test_loader))

# Find one sample of digit 1 and one of digit 7
idx_a = (test_labels == 1).nonzero(as_tuple=True)[0][0]
idx_b = (test_labels == 7).nonzero(as_tuple=True)[0][0]

x_a = test_images[idx_a].unsqueeze(0).to(device)  # (1, 1, 28, 28)
x_b = test_images[idx_b].unsqueeze(0).to(device)

with torch.no_grad():
    z_a = model.encode(x_a)  # (1, 2)
    z_b = model.encode(x_b)  # (1, 2)

# Linear interpolation: 10 steps from z_a to z_b
n_steps = 10
alphas  = torch.linspace(0, 1, n_steps)

fig, axes = plt.subplots(1, n_steps, figsize=(15, 2))

with torch.no_grad():
    for i, alpha in enumerate(alphas):
        z_interp = (1 - alpha) * z_a + alpha * z_b
        img = model.decoder(z_interp).squeeze().cpu()
        axes[i].imshow(img, cmap="gray")
        axes[i].axis("off")
        axes[i].set_title(f"{alpha:.1f}", fontsize=8)

plt.suptitle(f"Interpolation: digit {test_labels[idx_a].item()} → digit {test_labels[idx_b].item()}", y=1.05)
plt.tight_layout()
plt.show()